In [8]:
import tensorflow as tf
from tensorflow import keras
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

In [9]:
housing = fetch_california_housing()

In [10]:
#Split data
X_train_full, X_test, y_train_full, y_test = train_test_split( housing.data,
         housing.target.reshape(-1, 1), random_state=42)

X_train, X_valid, y_train, y_valid =  train_test_split( X_train_full, 
        y_train_full, random_state=42)

In [13]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [23]:
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)
X_test_scaled = scaler.transform(X_test)

# Custom Loss Function: Huber Loss

In machine learning, a **loss function** measures how wrong the model's predictions are compared to the true values.

The goal of training a model is to **minimize this loss**.

In this example, we implement a **custom Huber Loss function**.

---

# Code

```python
# Custom loss
def huber_fn(y_true, y_pred):
    error = y_true - y_pred
    is_small_error = tf.abs(error) < 1
    squared_loss = tf.square(error) / 2
    linear_loss  = tf.abs(error) - 0.5
    return tf.where(is_small_error, squared_loss, linear_loss)
```

---

# What is Huber Loss?

Huber Loss is a **combination of two popular regression loss functions**:

1. **Mean Squared Error (MSE)**
2. **Mean Absolute Error (MAE)**

It behaves like:

- **MSE for small errors**
- **MAE for large errors**

This makes it **more robust to outliers**.

---

# Why Not Use Only MSE?

Mean Squared Error squares the error:

```
Loss = (y_true - y_pred)²
```

This means large errors become **very large numbers**.

Example:

| Error | MSE |
|------|------|
| 2 | 4 |
| 10 | 100 |

So **outliers dominate the training process**.

---

# Why Not Use Only MAE?

MAE calculates:

```
Loss = |y_true - y_pred|
```

This handles outliers better, but it has a drawback:

- It is **not smooth for optimization**
- Gradients become unstable during training

---

# Huber Loss Solution

Huber loss combines the advantages of both.

For **small errors**:

```
Loss = error² / 2
```

For **large errors**:

```
Loss = |error| - 0.5
```

This means:

- small errors → behave like **MSE**
- large errors → behave like **MAE**

---

# Mathematical Definition

$$
L_\delta(a)=
\begin{cases}
\frac{1}{2}a^2 & \text{if } |a| \leq \delta \\
\delta(|a|-\frac{1}{2}\delta) & \text{otherwise}
\end{cases}
$$
Where:

- \(a = y_{true} - y_{pred}\)
- \(\delta\) is the threshold (in our code it is **1**)

---

# Step-by-Step Code Explanation

### Step 1 — Calculate Error

```python
error = y_true - y_pred
```

This calculates the **difference between the actual value and predicted value**.

Example:

```
True value = 10
Prediction = 8

error = 2
```

---

### Step 2 — Check if Error is Small

```python
is_small_error = tf.abs(error) < 1
```

Here we check whether the error is **smaller than the threshold (1)**.

If:

```
|error| < 1
```

we treat it as a **small error**.

---

### Step 3 — Compute Squared Loss

```python
squared_loss = tf.square(error) / 2
```

This calculates the **MSE style loss**.

```
loss = error² / 2
```

We divide by 2 because it simplifies the gradient during optimization.

---

### Step 4 — Compute Linear Loss

```python
linear_loss = tf.abs(error) - 0.5
```

This calculates the **MAE style loss** for large errors.

This prevents large errors from growing too fast.

---

### Step 5 — Choose the Correct Loss

```python
return tf.where(is_small_error, squared_loss, linear_loss)
```

`tf.where` acts like an **if-else statement for tensors**.

It means:

```
if error is small:
    use squared_loss
else:
    use linear_loss
```

---

# Visual Intuition

Huber loss curve looks like this:

```
        |
  Loss  |        /
        |       /
        |      /
        |_____/_____
              error
```

Near zero → **quadratic curve (MSE)**  
Far away → **linear line (MAE)**

---

# Why Huber Loss is Useful

Advantages:

- Robust to **outliers**
- Smooth gradients
- Stable training
- Combines **MSE + MAE advantages**

---

# When Huber Loss is Used

Huber loss is commonly used in:

- **Regression problems**
- **Object detection models**
- **Financial forecasting**
- **Robust ML models**

For example, it is used in **Fast R-CNN object detection models**.

---

# One Line Summary

**Huber Loss behaves like Mean Squared Error for small errors and Mean Absolute Error for large errors, making it robust to outliers while maintaining stable gradients during training.**

In [32]:
#Custom loss
def huber_fn(y_true, y_pred):
    error = y_true - y_pred
    is_small_error = tf.abs(error) < 1
    squared_loss = tf.square(error) / 2
    linear_loss  = tf.abs(error) - 0.5
    return tf.where(is_small_error, squared_loss, linear_loss)

In [35]:
#Define model
model = keras.models.Sequential([
    keras.layers.Dense(30, activation="selu", 
       kernel_initializer="lecun_normal", 
       input_shape=X_train.shape[1:]),
    keras.layers.Dense(1),
])

In [36]:
model.compile(loss=huber_fn, optimizer="nadam", metrics=["mae"])

In [39]:
model.fit(X_train_scaled, y_train, epochs=15,
  validation_data=(X_valid_scaled, y_valid))

Epoch 1/15
363/363 [==============================] - 1s 2ms/step - loss: 0.5688 - mae: 0.9284 - val_loss: 0.2156 - val_mae: 0.5039
Epoch 2/15
363/363 [==============================] - 1s 2ms/step - loss: 0.2041 - mae: 0.4947 - val_loss: 0.1919 - val_mae: 0.4791
Epoch 3/15
363/363 [==============================] - 1s 2ms/step - loss: 0.1990 - mae: 0.4879 - val_loss: 0.1864 - val_mae: 0.4692
Epoch 4/15
363/363 [==============================] - 1s 2ms/step - loss: 0.1956 - mae: 0.4829 - val_loss: 0.1902 - val_mae: 0.4716
Epoch 5/15
363/363 [==============================] - 1s 2ms/step - loss: 0.1924 - mae: 0.4786 - val_loss: 0.2016 - val_mae: 0.4836
Epoch 6/15
363/363 [==============================] - 1s 2ms/step - loss: 0.1901 - mae: 0.4751 - val_loss: 0.1977 - val_mae: 0.4752
Epoch 7/15
363/363 [==============================] - 1s 2ms/step - loss: 0.1887 - mae: 0.4720 - val_loss: 0.1790 - val_mae: 0.4601
Epoch 8/15
363/363 [==============================] - 1s 2ms/step - loss: 0.

In [40]:
model.evaluate(X_test_scaled, y_test)

162/162 [==============================] - 0s 1ms/step - loss: 0.1773 - mae: 0.4555


[0.17725634574890137, 0.455490380525589]